# 3.0 - Notebook Support Claimback Browser

HTML building for displaying the analysis (similar to the SQL browser)


In [9]:
from pathlib import Path
import re
import pandas as pd
from IPython.display import display
import ipywidgets as widgets

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_rows", 250)
pd.set_option("display.float_format", "{:,.2f}".format)


In [10]:
PROJECT_DIR = Path(".")
clean_data_dir = PROJECT_DIR / "outputs" / "cleaned_data"
support_cleaned_file = clean_data_dir / "support_cleaned.csv"
gl_accounts_cleaned_file = clean_data_dir / "gl_accounts_cleaned.csv"
gl_entries_cleaned_file = clean_data_dir / "gl_entries_cleaned.csv"
YEAR = 2026
START_DATE = pd.Timestamp(f"{YEAR}-01-01")
END_DATE = pd.Timestamp(f"{YEAR + 1}-01-01")
EXCLUDED_SUPPLIER_CODES = {"E005GB", "E059"}
MEDIUM_RISK_THRESHOLD = 2500.0
HIGH_RISK_THRESHOLD = 10000.0
print("Project folder:", PROJECT_DIR.resolve())
print("Cleaned support:", support_cleaned_file.exists(), "cleaned GL accounts:", gl_accounts_cleaned_file.exists(), "cleaned GL entries:", gl_entries_cleaned_file.exists())


Project folder: \\bpfile\product management\Product Management\(4) Cathal Heaney\Uni\8030
Cleaned support: True cleaned GL accounts: True cleaned GL entries: True


## Helper Functions


In [11]:
def read_cleaned_dataset(path):
    if not path.exists():
        raise FileNotFoundError(f"Cleaned dataset not found: {path}. Run 0.2 - Data Cleaning first.")
    return pd.read_csv(path, low_memory=False)


def supplier_group_code(code):
    if pd.isna(code): return ""
    code = str(code).strip()
    if code in {"L810", "L812"}: return "L810/L812"
    if code in {"E907", "E907GB"}: return "E907/E907GB"
    return code


def supplier_group_name(code, name):
    group = supplier_group_code(code)
    if group == "L810/L812": return "Supplier14"
    if group == "E907/E907GB": return "Supplier21"
    return "" if pd.isna(name) else str(name).strip()


def supplier_key(series):
    return series.astype("string").str.strip().str.upper().str.replace(" ", "", regex=False)


def fmt_money(value):
    value = 0.0 if pd.isna(value) else float(value)
    sign = "-" if value < 0 else ""
    return f"{sign}GBP {abs(value):,.2f}"


def risk_band(outstanding):
    outstanding = 0.0 if pd.isna(outstanding) else float(outstanding)
    if outstanding >= HIGH_RISK_THRESHOLD: return "High"
    if outstanding >= MEDIUM_RISK_THRESHOLD: return "Medium"
    if outstanding > 0: return "Low"
    return "Clear"


def show_df(df, money_cols=None, int_cols=None, date_cols=None, max_rows=350):
    money_cols, int_cols, date_cols = money_cols or [], int_cols or [], date_cols or []
    if df is None or df.empty:
        print("No rows for this view."); return
    out = df.head(max_rows).copy()
    for col in money_cols:
        if col in out: out[col] = out[col].map(fmt_money)
    for col in int_cols:
        if col in out: out[col] = out[col].fillna(0).map(lambda x: f"{float(x):,.0f}")
    for col in date_cols:
        if col in out: out[col] = pd.to_datetime(out[col], errors="coerce").dt.strftime("%Y-%m-%d").fillna("")
    display(out)
    if len(df) > max_rows: print(f"Showing first {max_rows:,} of {len(df):,} rows.")


## Load Data


In [12]:
def extract_supplier_code(description):
    text = "" if pd.isna(description) else str(description).upper()
    matches = re.findall(r"(?<![A-Z0-9])[A-Z]\d{3}(?:GB)?(?![A-Z0-9])", text)
    return supplier_group_code(matches[-1]) if matches else ""


def load_support_data():
    usecols = ["Date", "Customer", "Customer_Name", "Product", "Product_ManufacturerProductCode", "Product_Category", "Contract_Number", "SourceTransactionType", "Contract_D_ContractNumber", "SalesInvoice_Number", "SalesDeliveryNote_Number", "Status", "UnitClaimAmount", "SalesDeliveryNoteLine_Quantity", "SalesInvoiceLine_Quantity", "SalesDeliveryNote_NetAmountLessDiscountBase", "Contract_Description", "Contract_Expression", "Contract_ValidFrom", "Contract_ValidTo", "Contract_Supplier_Code", "Contract_Supplier_Name"]
    df = read_cleaned_dataset(support_cleaned_file)
    df = df[[col for col in usecols if col in df.columns]]
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce", dayfirst=True)
    df = df[df["Date"].ge(START_DATE) & df["Date"].lt(END_DATE) & df["SourceTransactionType"].eq("SL/Del") & ~df["Contract_Supplier_Code"].isin(EXCLUDED_SUPPLIER_CODES)].copy()
    df["SupplierCode"] = df["Contract_Supplier_Code"].map(supplier_group_code)
    df["SupplierName"] = [supplier_group_name(c, n) for c, n in zip(df["Contract_Supplier_Code"], df["Contract_Supplier_Name"])]
    df["SupplierKey"] = supplier_key(df["SupplierName"])
    df["Quantity"] = pd.to_numeric(df["SalesDeliveryNoteLine_Quantity"], errors="coerce")
    df["Quantity"] = df["Quantity"].fillna(pd.to_numeric(df["SalesInvoiceLine_Quantity"], errors="coerce")).fillna(0)
    df["UnitClaimAmount"] = pd.to_numeric(df["UnitClaimAmount"], errors="coerce").fillna(0)
    df["TotalSupportOwed"] = df["UnitClaimAmount"] * df["Quantity"]
    df["TotalNetAmountLessDiscountBase"] = pd.to_numeric(df["SalesDeliveryNote_NetAmountLessDiscountBase"], errors="coerce").fillna(0)
    df["Month"] = df["Date"].dt.to_period("M").dt.to_timestamp()
    return df


def load_ledger_data(support):
    accounts_raw = read_cleaned_dataset(gl_accounts_cleaned_file)
    entries_raw = read_cleaned_dataset(gl_entries_cleaned_file)
    accounts = pd.DataFrame({
        "GLAccountCode": accounts_raw.get("GL_Account_Code", accounts_raw.get("gl_account_code", accounts_raw.get("Code"))),
        "GLAccountDescription": accounts_raw.get("Description", accounts_raw.get("gl_account_description")),
        "CurrentBalance": pd.to_numeric(accounts_raw.get("Current_Balance", accounts_raw.get("gl_current_balance", accounts_raw.get("CurrentBalance"))), errors="coerce"),
    })
    entries = pd.DataFrame({
        "GLAccountCode": entries_raw.get("GL_Account_Code", entries_raw.get("gl_account_code")),
        "Description": entries_raw.get("Description", entries_raw.get("gl_account_description")),
        "LedgerDate": pd.to_datetime(entries_raw.get("Entry_Line_Date", entries_raw.get("gl_entry_line_date")), errors="coerce"),
        "CreditAmount": pd.to_numeric(entries_raw.get("Credit_Amount", entries_raw.get("gl_credit_amount")), errors="coerce").fillna(0),
        "RunningBalance": pd.to_numeric(entries_raw.get("Running_Balance"), errors="coerce"),
        "TotalBalance": pd.to_numeric(entries_raw.get("Total_Balance"), errors="coerce"),
    })
    accounts["SupplierCode"] = accounts["GLAccountDescription"].map(extract_supplier_code)
    ledger = entries.merge(accounts[["GLAccountCode", "GLAccountDescription", "SupplierCode", "CurrentBalance"]].drop_duplicates("GLAccountCode"), on="GLAccountCode", how="left")
    missing = ledger["SupplierCode"].isna() | ledger["SupplierCode"].eq("")
    ledger.loc[missing, "SupplierCode"] = ledger.loc[missing, "Description"].map(extract_supplier_code)
    supplier_lookup = support[["SupplierCode", "SupplierName"]].drop_duplicates("SupplierCode")
    ledger = ledger.merge(supplier_lookup, on="SupplierCode", how="left")
    ledger["SupplierCode"] = ledger["SupplierCode"].fillna("")
    ledger["SupplierName"] = ledger["SupplierName"].fillna(ledger["SupplierCode"])
    ledger = ledger[ledger["LedgerDate"].ge(START_DATE) & ledger["LedgerDate"].lt(END_DATE)].copy()
    ledger["Month"] = ledger["LedgerDate"].dt.to_period("M").dt.to_timestamp()
    return ledger


def build_summary(support, ledger):
    names = support.groupby("SupplierCode", as_index=False).agg(SupplierName=("SupplierName", "first"), LinkedSupplierCodes=("Contract_Supplier_Code", lambda x: ", ".join(sorted(set(map(str, x.dropna()))))))
    sup = support.groupby("SupplierCode", as_index=False).agg(LineCount=("Date", "size"), ContractCount=("Contract_D_ContractNumber", "nunique"), CustomerCount=("Customer", "nunique"), ProductCount=("Product", "nunique"), TotalQuantity=("Quantity", "sum"), TotalSupportOwed=("TotalSupportOwed", "sum"), FirstSupportDate=("Date", "min"), LastSupportDate=("Date", "max"))
    led = ledger[ledger["SupplierCode"].ne("")].groupby("SupplierCode", as_index=False).agg(LedgerPaid=("CreditAmount", "sum"), LatestLedgerDate=("LedgerDate", "max"), LedgerLineCount=("LedgerDate", "size"), GLAccounts=("GLAccountCode", lambda x: ", ".join(sorted(set(map(str, x.dropna()))))))
    summary = names.merge(sup, on="SupplierCode", how="left").merge(led, on="SupplierCode", how="left")
    summary["LedgerPaid"] = summary["LedgerPaid"].fillna(0)
    summary["Outstanding"] = summary["TotalSupportOwed"].fillna(0) - summary["LedgerPaid"]
    summary["RiskBand"] = summary["Outstanding"].map(risk_band)
    return summary.sort_values(["Outstanding", "TotalSupportOwed"], ascending=False)


def load_or_build():
    support = load_support_data()
    ledger = load_ledger_data(support)
    summary = build_summary(support, ledger)
    return support, ledger, summary


support, ledger, supplier_summary = load_or_build()
print("Support rows:", len(support), "Ledger rows:", len(ledger), "Supplier groups:", len(supplier_summary))
display(supplier_summary.head(10))


Support rows: 13464 Ledger rows: 100 Supplier groups: 19
Loaded cleaned datasets from: outputs\cleaned_data


,SupplierCode,SupplierName,LinkedSupplierCodes,LineCount,ContractCount,CustomerCount,ProductCount,TotalQuantity,TotalSupportOwed,FirstSupportDate,LastSupportDate,LedgerPaid,LatestLedgerDate,LedgerLineCount,GLAccounts,Outstanding,RiskBand
2,E070,Supplier19,E070,444,13,14,127,1144,"111,402.22",2026-01-05,2026-06-18,0.00,NaT,NaN,NaN,"111,402.22",High
3,E071,Supplier13,E071,699,29,106,21,1152,"257,950.37",2026-01-05,2026-06-18,"179,305.04",2026-03-07,82.00,01-85001,"78,645.33",High
17,U102,Supplier4,U102,197,6,62,16,251,"67,749.74",2026-01-05,2026-06-18,0.00,NaT,NaN,NaN,"67,749.74",High
18,U106,Supplier1,U106,7542,96,181,198,48173,"41,075.68",2026-01-05,2026-06-18,0.00,NaT,NaN,NaN,"41,075.68",High
8,E555,Supplier18,E555,1512,36,35,119,6082,"49,246.71",2026-01-05,2026-06-18,"9,662.89",2026-10-06,2.00,01-85079,"39,583.82",High
10,E907/E907GB,Supplier21,"E907, E907GB",138,7,55,24,160,"34,462.00",2026-01-05,2026-06-17,"17,397.00",2026-08-04,5.00,01-85185,"17,065.00",High
4,E102,Supplier7,E102,311,2,113,14,676,"7,571.96",2026-01-05,2026-06-18,0.00,NaT,NaN,NaN,"7,571.96",Medium
11,E948,Supplier20,E948,255,11,11,97,2155,"5,604.23",2026-01-05,2026-06-18,76.22,2026-07-04,1.00,01-85115,"5,528.01",Medium
13,L810/L812,Supplier14,"L810, L812",899,0,17,139,32470,"7,208.41",2026-01-05,2026-06-18,"2,002.26",2026-08-07,2.00,01-85137,"5,206.15",Medium
5,E175,Supplier8,E175,824,3,230,10,1634,"5,647.20",2026-01-05,2026-06-18,"2,459.31",2026-04-06,1.00,01-85036,"3,187.89",Medium


## Detail Functions


In [13]:
def contract_view(code):
    df = support[support["SupplierCode"].eq(code)]
    return df.groupby(["Contract_Number","Contract_D_ContractNumber","Contract_Description","Contract_Expression","Contract_ValidFrom","Contract_ValidTo"], dropna=False, as_index=False).agg(LineCount=("Date","size"), CustomerCount=("Customer","nunique"), ProductCount=("Product","nunique"), TotalQuantity=("Quantity","sum"), TotalSupportOwed=("TotalSupportOwed","sum")).sort_values("Contract_D_ContractNumber")

def customer_view(code):
    df = support[support["SupplierCode"].eq(code)]
    return df.groupby(["Customer","Customer_Name"], dropna=False, as_index=False).agg(LineCount=("Date","size"), ContractCount=("Contract_D_ContractNumber","nunique"), ProductCount=("Product","nunique"), TotalQuantity=("Quantity","sum"), TotalNetAmountLessDiscountBase=("TotalNetAmountLessDiscountBase","sum"), TotalSupportOwed=("TotalSupportOwed","sum")).sort_values("TotalSupportOwed", ascending=False)

def product_view(code):
    df = support[support["SupplierCode"].eq(code)]
    return df.groupby(["Product","Product_ManufacturerProductCode","Product_Category"], dropna=False, as_index=False).agg(LineCount=("Date","size"), ContractCount=("Contract_D_ContractNumber","nunique"), CustomerCount=("Customer","nunique"), TotalQuantity=("Quantity","sum"), TotalNetAmountLessDiscountBase=("TotalNetAmountLessDiscountBase","sum"), TotalSupportOwed=("TotalSupportOwed","sum")).sort_values(["Product_Category","Product"])

def monthly_view(code):
    sm = support[support["SupplierCode"].eq(code)].groupby("Month", as_index=False).agg(SupportLines=("Date","size"), Customers=("Customer","nunique"), Products=("Product","nunique"), Quantity=("Quantity","sum"), SupportOwed=("TotalSupportOwed","sum"))
    lm = ledger[ledger["SupplierCode"].eq(code)].groupby("Month", as_index=False).agg(LedgerLines=("LedgerDate","size"), LedgerPaid=("CreditAmount","sum"), LatestLedgerDate=("LedgerDate","max"))
    months = pd.DataFrame({"Month": pd.date_range(START_DATE, END_DATE - pd.offsets.MonthBegin(1), freq="MS")})
    out = months.merge(sm, on="Month", how="left").merge(lm, on="Month", how="left")
    for col in ["SupportLines","Customers","Products","Quantity","SupportOwed","LedgerLines","LedgerPaid"]: out[col] = out[col].fillna(0)
    out["OutstandingMovement"] = out["SupportOwed"] - out["LedgerPaid"]
    out["CumulativeOutstanding"] = out["OutstandingMovement"].cumsum()
    out["Month"] = out["Month"].dt.strftime("%Y-%m")
    return out

def ledger_view(code):
    cols = [c for c in ["LedgerDate","GLAccountCode","GLAccountDescription","Description","CreditAmount","RunningBalance"] if c in ledger]
    return ledger[ledger["SupplierCode"].eq(code)][cols].sort_values("LedgerDate", ascending=False)

def support_lines_view(code):
    cols = ["Date", "Customer", "Customer_Name", "Product", "Product_ManufacturerProductCode", "Product_Category", "Contract_D_ContractNumber", "Contract_Description", "Quantity", "UnitClaimAmount", "TotalSupportOwed", "TotalNetAmountLessDiscountBase", "SalesDeliveryNote_Number", "SalesInvoice_Number", "Status"]
    cols = [col for col in cols if col in support.columns]
    return support[support["SupplierCode"].eq(code)][cols].sort_values("Date", ascending=False)


## HTML Browser

Running this cell builds the browser page from the notebook tables and opens it in your default browser. Click a supplier row to update the KPIs, charts and monthly pivot table.


In [14]:
import json
import os
import tempfile
from pathlib import Path


def browser_records(df, limit=None):
    out = df.copy()
    if limit is not None:
        out = out.head(limit)
    for col in out.columns:
        if pd.api.types.is_datetime64_any_dtype(out[col]):
            out[col] = out[col].dt.strftime("%Y-%m-%d").fillna("")
        elif pd.api.types.is_period_dtype(out[col]):
            out[col] = out[col].astype(str)
    return json.loads(out.fillna("").to_json(orient="records", date_format="iso"))

summary_for_browser = supplier_summary.copy()
summary_for_browser["PaidPercent"] = 0.0
if "TotalSupportOwed" in summary_for_browser and "LedgerPaid" in summary_for_browser:
    owed = pd.to_numeric(summary_for_browser["TotalSupportOwed"], errors="coerce").fillna(0)
    paid = pd.to_numeric(summary_for_browser["LedgerPaid"], errors="coerce").fillna(0)
    summary_for_browser["PaidPercent"] = (paid / owed.replace(0, pd.NA) * 100).fillna(0).round(1)
for col in ["TotalSupportOwed", "LedgerPaid", "Outstanding"]:
    if col in summary_for_browser:
        summary_for_browser[col] = pd.to_numeric(summary_for_browser[col], errors="coerce").round(2)

supplier_data = {}
for code in supplier_summary["SupplierCode"].dropna().astype(str):
    supplier_data[code] = {
        "monthly": browser_records(monthly_view(code)),
        "contracts": browser_records(contract_view(code), 80),
        "customers": browser_records(customer_view(code), 80),
        "products": browser_records(product_view(code), 80),
        "ledger": browser_records(ledger_view(code), 80),
        "support_lines": browser_records(support_lines_view(code), 120),
    }

page_data = {
    "summary": browser_records(summary_for_browser),
    "suppliers": supplier_data,
}

html_page = """
<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>Claimback Notebook Browser</title>
<style>
body {
  margin: 0;
  font: 12px Arial, sans-serif;
  color: #202020;
  background: #f4f5f7;
}
header {
  background: #26364a;
  color: white;
  padding: 8px 14px;
}
header h1 {
  margin: 0;
  font-size: 16px;
}
.controls {
  display: flex;
  gap: 10px;
  align-items: center;
  padding: 8px 14px;
  background: white;
  border-bottom: 1px solid #d0d5dd;
  position: sticky;
  top: 0;
  z-index: 5;
}
.controls .hint {
  color: #667085;
}
input {
  padding: 6px 8px;
  border: 1px solid #aeb6c2;
  border-radius: 4px;
  background: white;
  min-width: 260px;
}
main {
  padding: 8px 10px 12px;
}
.top-panel {
  display: grid;
  grid-template-columns: minmax(560px, 0.9fr) minmax(620px, 1.25fr);
  gap: 8px;
  margin-bottom: 6px;
}
.kpis {
  display: grid;
  grid-template-columns: repeat(4, minmax(150px, 1fr));
  gap: 8px;
}
.kpi {
  background: white;
  border: 1px solid #d0d5dd;
  padding: 9px 10px;
  border-radius: 4px;
  min-width: 0;
  min-height: 76px;
}
.kpi.active {
  border-color: #26364a;
  box-shadow: inset 4px 0 0 #26364a;
}
.kpi span {
  display: block;
  color: #5b6573;
  font-size: 10px;
  text-transform: uppercase;
}
.kpi strong {
  display: block;
  font-size: 15px;
  margin-top: 4px;
  white-space: nowrap;
  overflow: hidden;
  text-overflow: ellipsis;
}
.charts {
  display: grid;
  grid-template-columns: 0.8fr 1.2fr;
  gap: 8px;
}
.chartbox {
  background: white;
  border: 1px solid #d0d5dd;
  border-radius: 4px;
  padding: 8px 10px;
  min-height: 116px;
}
.chartbox h3 {
  margin: 0 0 5px;
  font-size: 12px;
  color: #344054;
}
canvas {
  width: 100%;
  height: 86px;
  display: block;
}
section {
  background: white;
  border: 1px solid #d0d5dd;
  border-radius: 4px;
  margin-bottom: 10px;
  overflow: hidden;
}
h2 {
  font-size: 13px;
  margin: 0;
  padding: 7px 10px;
  background: #eef1f4;
  border-bottom: 1px solid #d0d5dd;
}
.table-wrap {
  max-height: 340px;
  overflow: auto;
  position: relative;
}
.summary-table {
  margin-bottom: 4px;
}
.summary-table .table-wrap {
  max-height: 430px;
}
.detail-panel {
  margin-top: 0;
  margin-bottom: 0;
}
.detail-panel > section {
  border: 0;
  border-radius: 0;
  margin: 0;
}
.detail-panel .table-wrap {
  max-height: 150px;
}
table {
  border-collapse: separate;
  border-spacing: 0;
  min-width: max-content;
  width: 100%;
}
th,
td {
  border-bottom: 1px solid #e1e5ea;
  padding: 4px 8px;
  text-align: left;
  vertical-align: top;
  white-space: nowrap;
}
th {
  background: #f8f9fb;
  position: sticky;
  top: 0;
  z-index: 3;
  box-shadow: inset 0 -1px 0 #cbd2dc;
  font-weight: 700;
}
td.num,
th.num,
td.money,
th.money,
td.percent,
th.percent {
  text-align: right;
  font-variant-numeric: tabular-nums;
}
td.date,
th.date {
  text-align: center;
  font-variant-numeric: tabular-nums;
}
tbody tr:nth-child(even) {
  background: #fbfcfe;
}
tbody tr:hover {
  background: #fff7df;
  cursor: pointer;
}
tr.selected {
  background: #eaf2ff !important;
  box-shadow: inset 3px 0 0 #26364a;
}
.badge {
  display: inline-block;
  padding: 2px 7px;
  border-radius: 10px;
  background: #e8eef7;
  color: #1d2939;
  font-size: 11px;
  margin-left: 6px;
}
.tabs {
  display: flex;
  flex-wrap: wrap;
  gap: 0;
  background: #eef1f4;
  border-bottom: 1px solid #d0d5dd;
}
.tab {
  border: 0;
  border-right: 1px solid #d0d5dd;
  background: #eef1f4;
  padding: 6px 12px;
  font: inherit;
  font-weight: 700;
  cursor: pointer;
  color: #344054;
}
.tab:hover {
  background: #f8f9fb;
}
.tab.active {
  background: white;
  color: #172033;
  box-shadow: inset 0 3px 0 #26364a;
}
.detail-panel h2 {
  display: none;
}
@media (max-width: 1050px) {
  .top-panel {
    grid-template-columns: 1fr;
  }
  .charts {
    grid-template-columns: 1fr 1fr;
  }
}
@media (max-width: 760px) {
  .kpis,
  .charts {
    grid-template-columns: 1fr;
  }
  .controls {
    flex-wrap: wrap;
  }
}
</style>
</head>
<body>
<header><h1>Claimback Notebook Browser</h1></header>
<div class="controls">
<input id="searchBox" placeholder="Search the visible tables">
</div>
<main id="app"></main>
<script>
const DATA = __DATA__;
const app = document.getElementById('app');
const searchBox = document.getElementById('searchBox');
let selectedCode = DATA.summary[0]?.SupplierCode || '';
let activeTab = 'monthly';

function money(value) {
  const n = Number(value || 0);
  return '£' + n.toLocaleString(undefined, {minimumFractionDigits: 2, maximumFractionDigits: 2});
}
function prettyName(name) {
  return String(name).replace(/_/g, ' ').replace(/([a-z])([A-Z])/g, '$1 $2');
}
function colType(col) {
  if (/date|month/i.test(col)) return 'date';
  if (/percent|%/i.test(col)) return 'percent';
  if (/owed|paid|outstanding|amount|value|balance/i.test(col)) return 'money';
  if (/count|lines|quantity|customers|products|contracts|rows/i.test(col)) return 'num';
  return '';
}
function formatCell(col, value) {
  if (value === null || value === undefined || value === '' || String(value) === 'NaN') return '';
  const type = colType(col);
  if (type === 'money') return money(value);
  if (type === 'percent' && !isNaN(Number(value))) return Number(value).toLocaleString(undefined, {minimumFractionDigits: 1, maximumFractionDigits: 1}) + '%';
  if (type === 'num' && !isNaN(Number(value))) return Number(value).toLocaleString();
  return String(value);
}
function filterRows(rows) {
  const q = searchBox.value.toLowerCase();
  return q ? rows.filter(r => Object.values(r).join(' ').toLowerCase().includes(q)) : rows;
}
function makeTable(title, rows, opts = {}) {
  const className = opts.className ? ` class="${opts.className}"` : '';
  if (!rows || rows.length === 0) return `<section${className}><h2>${title}</h2><p style="padding:10px 12px">No rows.</p></section>`;
  const filtered = filterRows(rows);
  const cols = Object.keys(rows[0]);
  const head = cols.map(c => `<th class="${colType(c)}">${prettyName(c)}</th>`).join('');
  const body = filtered.map(r => {
    const code = String(r.SupplierCode || '');
    const selected = opts.clickSupplier && code === String(selectedCode) ? ' class="selected"' : '';
    const attr = opts.clickSupplier ? ` data-supplier="${code}"` : '';
    return `<tr${selected}${attr}>${cols.map(c => `<td class="${colType(c)}">${formatCell(c, r[c])}</td>`).join('')}</tr>`;
  }).join('');
  return `<section${className}><h2>${title}<span class="badge">${filtered.length} rows</span></h2><div class="table-wrap"><table><thead><tr>${head}</tr></thead><tbody>${body}</tbody></table></div></section>`;
}
function monthlyPivot(rows) {
  const metrics = [
    ['SupportLines', 'Support Lines'],
    ['SupportOwed', 'Support Owed'],
    ['OutstandingMovement', 'Outstanding Movement'],
    ['CumulativeOutstanding', 'Cumulative Outstanding']
  ];
  const months = rows.map(r => r.Month);
  const pivotRows = metrics.map(([key, label]) => {
    const row = {Metric: label};
    rows.forEach(r => row[r.Month] = key === 'SupportLines' ? (r[key] ?? '') : Number(r[key] || 0).toFixed(2));
    return row;
  });
  return makeTable('Monthly Pivot', pivotRows, {className: 'detail-panel'});
}
function drawBars(canvas, selected) {
  const ctx = canvas.getContext('2d');
  const dpr = window.devicePixelRatio || 1;
  const rect = canvas.getBoundingClientRect();
  canvas.width = rect.width * dpr; canvas.height = rect.height * dpr; ctx.scale(dpr, dpr);
  ctx.clearRect(0, 0, rect.width, rect.height);
  const vals = [Number(selected.TotalSupportOwed||0), Number(selected.LedgerPaid||0), Number(selected.Outstanding||0)];
  const labels = ['Owed', 'Paid', 'Out'];
  const colors = ['#4978b8', '#3f8f5f', '#b94b4b'];
  const max = Math.max(...vals, 1);
  vals.forEach((v, i) => {
    const y = 18 + i * 30;
    ctx.fillStyle = '#667085'; ctx.fillText(labels[i], 0, y + 12);
    ctx.fillStyle = '#e6eaf0'; ctx.fillRect(42, y, rect.width - 52, 16);
    ctx.fillStyle = colors[i]; ctx.fillRect(42, y, (rect.width - 52) * (v / max), 16);
  });
}
function drawMonthly(canvas, rows) {
  const ctx = canvas.getContext('2d');
  const dpr = window.devicePixelRatio || 1;
  const rect = canvas.getBoundingClientRect();
  canvas.width = rect.width * dpr; canvas.height = rect.height * dpr; ctx.scale(dpr, dpr);
  ctx.clearRect(0, 0, rect.width, rect.height);
  const owed = rows.map(r => Number(r.SupportOwed || 0));
  const paid = rows.map(r => Number(r.LedgerPaid || 0));
  const max = Math.max(...owed, ...paid, 1);
  const w = Math.max(10, (rect.width - 30) / Math.max(rows.length, 1));
  rows.forEach((r, i) => {
    const x = 24 + i * w;
    const oh = (rect.height - 28) * owed[i] / max;
    const ph = (rect.height - 28) * paid[i] / max;
    ctx.fillStyle = '#4978b8'; ctx.fillRect(x, rect.height - 18 - oh, w * 0.35, oh);
    ctx.fillStyle = '#3f8f5f'; ctx.fillRect(x + w * 0.38, rect.height - 18 - ph, w * 0.35, ph);
    ctx.fillStyle = '#667085'; ctx.fillText(String(r.Month).slice(5), x, rect.height - 4);
  });
}
const detailTabs = [
  ['monthly', 'Monthly Pivot'],
  ['contracts', 'Contracts'],
  ['customers', 'Customers'],
  ['products', 'Products'],
  ['ledger', 'Ledger Lines'],
  ['support_lines', 'Support Lines']
];
function tabButtons() {
  return `<div class="tabs">${detailTabs.map(([key, label]) => `<button class="tab ${key === activeTab ? 'active' : ''}" data-tab="${key}">${label}</button>`).join('')}</div>`;
}
function detailTable(details) {
  if (activeTab === 'monthly') return monthlyPivot(details.monthly || []);
  const labels = Object.fromEntries(detailTabs);
  return makeTable(labels[activeTab] || 'Detail', details[activeTab] || [], {className: 'detail-panel'});
}
function attachTabClicks() {
  document.querySelectorAll('button[data-tab]').forEach(button => {
    button.addEventListener('click', () => {
      activeTab = button.dataset.tab;
      render();
    });
  });
}
function attachSupplierClicks() {
  document.querySelectorAll('tr[data-supplier]').forEach(row => {
    row.addEventListener('click', () => {
      selectedCode = row.dataset.supplier;
      render();
      window.scrollTo({top: 0, behavior: 'smooth'});
    });
  });
}
function render() {
  const selected = DATA.summary.find(row => String(row.SupplierCode) === String(selectedCode)) || DATA.summary[0];
  selectedCode = selected.SupplierCode;
  const details = DATA.suppliers[selectedCode] || {};
  app.innerHTML = `
    <div class="top-panel">
      <div class="kpis">
        <div class="kpi active"><span>Selected supplier</span><strong>${selected.SupplierCode} - ${selected.SupplierName || ''}</strong></div>
        <div class="kpi"><span>Paid %</span><strong>${formatCell('PaidPercent', selected.PaidPercent)}</strong></div>
        <div class="kpi"><span>Support owed</span><strong>${money(selected.TotalSupportOwed)}</strong></div>
        <div class="kpi"><span>Outstanding</span><strong>${money(selected.Outstanding)}</strong></div>
      </div>
      <div class="charts">
        <div class="chartbox"><h3>Position</h3><canvas id="positionChart"></canvas></div>
        <div class="chartbox"><h3>Monthly Owed vs Paid</h3><canvas id="monthlyChart"></canvas></div>
      </div>
    </div>
    ${makeTable('Supplier Summary', DATA.summary, {clickSupplier: true, className: 'summary-table'})}
    <section class="detail-panel">
      ${tabButtons()}
      ${detailTable(details)}
    </section>
  `;
  attachSupplierClicks();
  attachTabClicks();
  drawBars(document.getElementById('positionChart'), selected);
  drawMonthly(document.getElementById('monthlyChart'), details.monthly || []);
}
searchBox.addEventListener('input', render);
render();
</script>
</body>
</html>
""".replace("__DATA__", json.dumps(page_data, default=str))

temp_path = Path(tempfile.gettempdir()) / "claimback_notebook_browser.html"
temp_path.write_text(html_page, encoding="utf-8")
os.startfile(str(temp_path))
print("Opened claimback browser page:", temp_path)

C:\Users\cathalhe\AppData\Local\Temp\ipykernel_45124\2591493903.py:14: DeprecationWarning: is_period_dtype is deprecated and will be removed in a future version. Use `isinstance(dtype, pd.PeriodDtype)` instead
  elif pd.api.types.is_period_dtype(out[col]):
C:\Users\cathalhe\AppData\Local\Temp\ipykernel_45124\2591493903.py:14: DeprecationWarning: is_period_dtype is deprecated and will be removed in a future version. Use `isinstance(dtype, pd.PeriodDtype)` instead
  elif pd.api.types.is_period_dtype(out[col]):
C:\Users\cathalhe\AppData\Local\Temp\ipykernel_45124\2591493903.py:14: DeprecationWarning: is_period_dtype is deprecated and will be removed in a future version. Use `isinstance(dtype, pd.PeriodDtype)` instead
  elif pd.api.types.is_period_dtype(out[col]):
C:\Users\cathalhe\AppData\Local\Temp\ipykernel_45124\2591493903.py:14: DeprecationWarning: is_period_dtype is deprecated and will be removed in a future version. Use `isinstance(dtype, pd.PeriodDtype)` instead
  elif pd.api.types

Opened claimback browser page: C:\Users\cathalhe\AppData\Local\Temp\claimback_notebook_browser.html
